In [ ]:
!python --version
!git clone https://github.com/AleezaMujahid/AIES_PROJECT
%cd AIES_PROJECT
!pip -q install -r requirements.txt
!python Scripts/predict_baseline.py --model-dir models/baseline_generic --code "void f(char *s){ char b[8]; strcpy(b, s); }"

Python 3.13.16
Cloning into 'AIES_PROJECT'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 82 (delta 13), reused 77 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 51.97 MiB | 21.38 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/AIES_PROJECT
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: 

In [ ]:
!ln -s DATA data
!ls data/processed

bigvul_test.csv.gz   cvefixes_external.csv.gz	      split_manifest.csv
bigvul_train.csv.gz  cvefixes_external_strict.csv.gz
bigvul_val.csv.gz    cvefixes_review_sheet.csv


In [ ]:
!pip -q install transformers accelerate
import pandas as pd, torch, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification

df = pd.read_csv("data/processed/bigvul_train.csv.gz",
                 usecols=["code","label"]).sample(256, random_state=0)

tok = AutoTokenizer.from_pretrained("microsoft/codebert-base")
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base", num_labels=2).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler("cuda")

BS = 16
t0 = time.time()
for i in range(0, len(df), BS):
    b = df.iloc[i:i+BS]
    enc = tok(list(b.code.astype(str)), truncation=True, max_length=512,
              padding=True, return_tensors="pt").to("cuda")
    y = torch.tensor(b.label.astype(int).values).cuda()
    with torch.autocast("cuda", dtype=torch.float16):
        loss = torch.nn.functional.cross_entropy(model(**enc).logits, y)
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
dt = time.time() - t0
print(f"{len(df)/dt:.1f} samples/sec | loss {loss.item():.3f} | "
      f"GPU mem {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

... (3 outputs left out)

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

36.4 samples/sec | loss 0.236 | GPU mem 5.5 GB


In [ ]:
!python Scripts/07_top_tokens.py --model-dir models/baseline_generic --report reports/baseline_generic_top_tokens.md
!cat reports/baseline_generic_top_tokens.md

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
# Baseline: most influential tokens

Generated by Scripts/07_top_tokens.py. A larger positive weight pushes a function towards 'vulnerable'; a negative weight pushes it towards 'safe'. Tokens made of two words are pairs that appear next to each other.

| Rank | Towards vulnerable | Weight | Towards safe | Weight |
|---|---|---|---|---|
| 1 | `) ,` | +2.28 | `ID` | -2.35 |
| 2 | `; ID` | +1.97 | `static` | -1.94 |
| 3 | `virtual` | +1.91 | `Document :` | -1.92 |
| 4 | `( )` | +1.84 | `Document` | -1.65 |
| 5 | `header` | +1.77 | `return (` | -1.61 |
| 6 | `ID )` | +1.76 | `return` | -1.45 |
| 7 | `/ /` | +1.72 | `)` | -1.44 |
| 8 | `} if` | +1.65 | `(` | -1.44 |
| 9 | `<<` | +1.62 | `;` | -1.38 |
| 10 | `pm` | +1.59 | `struct` | -1.35 |
| 11 | `cc` | +1.54 | `( struct` | -1.33 |
|

In [ ]:
#Cell 1
from google.colab import drive
drive.mount('/content/drive')
import os, torch
if not os.path.exists('/content/AIES_PROJECT'):
    !git clone https://github.com/AleezaMujahid/AIES_PROJECT /content/AIES_PROJECT
%cd /content/AIES_PROJECT
if not os.path.exists('data'):
    !ln -s DATA data
!pip -q install transformers accelerate
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

Mounted at /content/drive
Cloning into '/content/AIES_PROJECT'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 82 (delta 13), reused 77 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 51.97 MiB | 35.06 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/AIES_PROJECT
True Tesla T4


In [ ]:
#CELL 2
SMOKE   = False      # True = tiny test. Set False for the real run.
MAX_LEN = 512
N_SAFE  = 40000
EPOCHS  = 2
LR      = 2e-5
BS      = 16
SEED    = 42
OUT = "/content/drive/MyDrive/codesentinel_ckpt_v2" + ("_smoke" if SMOKE else "")
os.makedirs(OUT, exist_ok=True)
print("SMOKE" if SMOKE else "FULL RUN", "->", OUT)

FULL RUN -> /content/drive/MyDrive/codesentinel_ckpt_v2


In [ ]:
#CELL 3
import numpy as np, pandas as pd, time, json, random, shutil
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

def evaluate(y_true, p_vuln, threshold=0.5):
    y_true = np.asarray(y_true); p = np.asarray(p_vuln)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"threshold": float(threshold), "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "pr_auc": average_precision_score(y_true, p), "roc_auc": roc_auc_score(y_true, p),
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

def best_f1_threshold(y_true, p):          # use on VAL only
    ths = np.linspace(0.01, 0.99, 99)
    f1s = [f1_score(y_true, (np.asarray(p) >= t).astype(int), zero_division=0) for t in ths]
    return float(ths[int(np.argmax(f1s))])

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cols = ["code", "label"]
train_full = pd.read_csv("data/processed/bigvul_train.csv.gz", usecols=cols)
val        = pd.read_csv("data/processed/bigvul_val.csv.gz",   usecols=cols)
train_full["label"] = train_full["label"].astype(int); val["label"] = val["label"].astype(int)
import re
def normalize_code(c):      # per-line strip, collapse spaces/tabs, drop blank lines
    return "\n".join(re.sub(r"[ \t]+", " ", l.strip()) for l in str(c).split("\n") if l.strip())
train_full["code"] = train_full["code"].map(normalize_code)
val["code"] = val["code"].map(normalize_code)
vul  = train_full[train_full.label == 1]
safe = train_full[train_full.label == 0].sample(min(N_SAFE, int((train_full.label == 0).sum())), random_state=SEED)
train = pd.concat([vul, safe]).sample(frac=1, random_state=SEED).reset_index(drop=True)
if SMOKE:
    train = train.sample(800, random_state=SEED).reset_index(drop=True)
    val   = val.sample(1000, random_state=SEED).reset_index(drop=True)
    EPOCHS = 1

n1 = int(train.label.sum()); n0 = len(train) - n1
class_w = torch.tensor([len(train)/(2*n0), len(train)/(2*n1)], dtype=torch.float).cuda()
print(f"full train rows {len(train_full):,} | used {len(train):,} ({n1:,} vulnerable, {n0:,} safe) | val rows {len(val):,}")
print("class weights [safe, vulnerable]:", class_w.tolist())

tok = AutoTokenizer.from_pretrained("microsoft/codebert-base")
torch.manual_seed(SEED)                                   # same random new layer every time
model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2).cuda()
torch.save(model.classifier.state_dict(), f"{OUT}/classifier_start.pt")

def batch_enc(codes):
    return tok(list(codes), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")

@torch.no_grad()
def score(df, bs=64):                                     # model score for class 1 (vulnerable)
    model.eval()
    order = np.argsort(df.code.str.len().values)
    out = np.zeros(len(df))
    for i in range(0, len(df), bs):
        idx = order[i:i+bs]
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**batch_enc(df.code.astype(str).values[idx])).logits
        out[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return out

rows = []
def log_rows(stage, s):
    y = val.label.values
    rows.append({"stage": stage, "threshold_type": "fixed 0.5", **evaluate(y, s, 0.5)})
    t = best_f1_threshold(y, s)
    rows.append({"stage": stage, "threshold_type": "best F1 on val", **evaluate(y, s, t)})
    pd.DataFrame(rows).to_csv(f"{OUT}/val_before_after.csv", index=False)

# BEFORE training (epoch 0)
s0 = score(val); np.save(f"{OUT}/val_scores_epoch0.npy", s0); log_rows("CodeBERT before (epoch 0)", s0)
print(pd.DataFrame(rows)[["stage","threshold_type","accuracy","precision","recall","f1","pr_auc"]].round(3))

full train rows 129,360 | used 46,748 (6,748 vulnerable, 40,000 safe) | val rows 27,720
class weights [safe, vulnerable]: [0.5843499898910522, 3.463841199874878]


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

... (3 outputs left out)

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

                       stage  threshold_type  accuracy  precision  recall     f1  pr_auc
0  CodeBERT before (epoch 0)       fixed 0.5     0.948      0.000   0.000  0.000   0.046
1  CodeBERT before (epoch 0)  best F1 on val     0.127      0.053   0.938  0.101   0.046


In [ ]:
#CELL 4
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps_total = EPOCHS * ((len(train) + BS - 1) // BS)
sched = get_linear_schedule_with_warmup(opt, int(0.06*steps_total), steps_total)
scaler = torch.amp.GradScaler("cuda")
best_ap, best_epoch = -1.0, None

for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); perm = np.random.permutation(len(train)); run = 0.0
    for n, i in enumerate(range(0, len(train), BS), 1):
        b = train.iloc[perm[i:i+BS]]
        enc = batch_enc(b.code.astype(str).values)
        y = torch.tensor(b.label.values).cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = torch.nn.functional.cross_entropy(model(**enc).logits.float(), y, weight=class_w)
        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
        if n % 200 == 0: print(f"epoch {epoch} step {n} avg loss {run/n:.4f} | {time.time()-t0:.0f}s")
    s = score(val); np.save(f"{OUT}/val_scores_epoch{epoch}.npy", s)
    log_rows(f"CodeBERT after epoch {epoch}", s)
    ap = average_precision_score(val.label, s)
    print(f"epoch {epoch}: val PR-AUC {ap:.4f} | {(time.time()-t0)/60:.1f} min")
    print(pd.DataFrame(rows)[["stage","threshold_type","accuracy","precision","recall","f1","pr_auc"]].round(3).tail(2))
    d = f"{OUT}/epoch_{epoch}"; model.save_pretrained(d); tok.save_pretrained(d)   # checkpoint EVERY epoch
    if ap > best_ap:
        best_ap, best_epoch = ap, epoch
        shutil.rmtree(f"{OUT}/best", ignore_errors=True); shutil.copytree(d, f"{OUT}/best")
print("best epoch", best_epoch, "val PR-AUC", best_ap)

epoch 1 step 200 avg loss 0.6702 | 91s
epoch 1 step 400 avg loss 0.6606 | 186s
epoch 1 step 600 avg loss 0.6559 | 281s
epoch 1 step 800 avg loss 0.6489 | 376s
epoch 1 step 1000 avg loss 0.6406 | 471s
epoch 1 step 1200 avg loss 0.6372 | 566s
epoch 1 step 1400 avg loss 0.6301 | 660s
epoch 1 step 1600 avg loss 0.6221 | 756s
epoch 1 step 1800 avg loss 0.6155 | 850s
epoch 1 step 2000 avg loss 0.6094 | 945s
epoch 1 step 2200 avg loss 0.6063 | 1039s
epoch 1 step 2400 avg loss 0.6006 | 1134s
epoch 1 step 2600 avg loss 0.5966 | 1229s
epoch 1 step 2800 avg loss 0.5933 | 1323s
epoch 1: val PR-AUC 0.1478 | 25.2 min
                    stage  threshold_type  accuracy  precision  recall     f1  pr_auc
2  CodeBERT after epoch 1       fixed 0.5     0.790      0.137   0.571  0.221   0.148
3  CodeBERT after epoch 1  best F1 on val     0.871      0.172   0.387  0.239   0.148


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 step 200 avg loss 0.4761 | 96s
epoch 2 step 400 avg loss 0.4853 | 191s
epoch 2 step 600 avg loss 0.4784 | 285s
epoch 2 step 800 avg loss 0.4741 | 380s
epoch 2 step 1000 avg loss 0.4706 | 474s
epoch 2 step 1200 avg loss 0.4703 | 568s
epoch 2 step 1400 avg loss 0.4686 | 663s
epoch 2 step 1600 avg loss 0.4683 | 758s
epoch 2 step 1800 avg loss 0.4639 | 852s
epoch 2 step 2000 avg loss 0.4614 | 947s
epoch 2 step 2200 avg loss 0.4593 | 1041s
epoch 2 step 2400 avg loss 0.4561 | 1135s
epoch 2 step 2600 avg loss 0.4566 | 1230s
epoch 2 step 2800 avg loss 0.4545 | 1325s
epoch 2: val PR-AUC 0.1562 | 25.3 min
                    stage  threshold_type  accuracy  precision  recall     f1  pr_auc
4  CodeBERT after epoch 2       fixed 0.5     0.829      0.154   0.508  0.237   0.156
5  CodeBERT after epoch 2  best F1 on val     0.877      0.181   0.385  0.246   0.156


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

best epoch 2 val PR-AUC 0.15616740358850276


In [ ]:
#CELL 5
s = np.load(f"{OUT}/val_scores_epoch{best_epoch}.npy"); y = val.label.values
thr = best_f1_threshold(y, s)                              # frozen threshold, found on VAL only
cfg = {"model": "microsoft/codebert-base", "max_length": MAX_LEN, "truncation": "keep start",
       "best_epoch": best_epoch, "epochs": EPOCHS, "lr": LR, "batch_size": BS, "seed": SEED, "smoke": SMOKE,
       "train_rows_full": int(len(train_full)), "train_rows_used": int(len(train)),
       "train_vulnerable_used": n1, "train_safe_used": n0, "class_weights": class_w.tolist(),
       "label_mapping": {"0": "not flagged", "1": "potentially vulnerable"},
       "threshold": thr, "val_at_threshold": evaluate(y, s, thr)}
cfg["preprocessing"] = "normalize_code: per-line strip, collapse spaces/tabs, drop blank lines"
json.dump(cfg, open(f"{OUT}/best/codebert_config.json", "w"), indent=2)
print(json.dumps(cfg, indent=2))
print(pd.read_csv(f"{OUT}/val_before_after.csv")[["stage","threshold_type","accuracy","precision","recall","f1","pr_auc"]].round(3))

{
  "model": "microsoft/codebert-base",
  "max_length": 512,
  "truncation": "keep start",
  "best_epoch": 2,
  "epochs": 2,
  "lr": 2e-05,
  "batch_size": 16,
  "seed": 42,
  "smoke": false,
  "train_rows_full": 129360,
  "train_rows_used": 46748,
  "train_vulnerable_used": 6748,
  "train_safe_used": 40000,
  "class_weights": [
    0.5843499898910522,
    3.463841199874878
  ],
  "label_mapping": {
    "0": "not flagged",
    "1": "potentially vulnerable"
  },
  "threshold": 0.66,
  "val_at_threshold": {
    "threshold": 0.66,
    "accuracy": 0.8771284271284271,
    "precision": 0.1811971372804164,
    "recall": 0.3852005532503458,
    "f1": 0.24646017699115044,
    "pr_auc": 0.15616740358850276,
    "roc_auc": 0.7678993958865877,
    "TN": 23757,
    "FP": 2517,
    "FN": 889,
    "TP": 557
  },
  "preprocessing": "normalize_code: per-line strip, collapse spaces/tabs, drop blank lines"
}
                       stage  threshold_type  accuracy  precision  recall     f1  pr_auc
0  CodeB

In [ ]:
from sklearn.metrics import precision_recall_curve
s = np.load(f"{OUT}/val_scores_epoch2.npy"); y = val.label.values
p, r, t = precision_recall_curve(y, s)
f1 = 2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1], 1e-12)
k = int(np.argmax(f1)); thr = float(t[k])
print("threshold", thr, "| F1", f1[k], "| P", p[k], "| R", r[k])
print("share of val scores >=0.99:", (s>=0.99).mean(), "| >=0.999:", (s>=0.999).mean())
res = evaluate(y, s, thr); print(res)
cfg["threshold"] = thr; cfg["val_at_threshold"] = res
json.dump(cfg, open(f"{OUT}/best/codebert_config.json", "w"), indent=2)

threshold 0.9943471550941467 | F1 0.8122866894197952 | P 0.8992443324937027 | R 0.7406639004149378
share of val scores >=0.99: 0.04498556998556998 | >=0.999: 0.03520923520923521
{'threshold': 0.9943471550941467, 'accuracy': 0.9821428571428571, 'precision': 0.8992443324937027, 'recall': 0.7406639004149378, 'f1': 0.8122866894197952, 'pr_auc': 0.863840200505478, 'roc_auc': 0.9741048584599092, 'TN': 26154, 'FP': 120, 'FN': 375, 'TP': 1071}


In [ ]:
import numpy as np, pandas as pd, time
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
from sklearn.feature_extraction.text import TfidfVectorizer

OUT = "/content/drive/MyDrive/codesentinel_ckpt"
THR = 0.9943471550941467                      # frozen val threshold
TOK = r"[A-Za-z_][A-Za-z0-9_]*|\d+|->|==|!=|<=|>=|\+\+|--|&&|\|\||<<|>>|[^\sA-Za-z0-9_]"

cols = ["code", "label", "project_key"]
tr = pd.read_csv("data/processed/bigvul_train.csv.gz", usecols=cols)
va = pd.read_csv("data/processed/bigvul_val.csv.gz",   usecols=cols)
s2 = np.load(f"{OUT}/val_scores_epoch2.npy"); assert len(s2) == len(va)

# all 1,446 vulnerable val functions + 3,000 random safe ones
rng = np.random.RandomState(42)
sel = np.concatenate([np.where(va.label.values == 1)[0],
                      rng.choice(np.where(va.label.values == 0)[0], 3000, replace=False)])

vec = TfidfVectorizer(token_pattern=TOK, lowercase=False, max_df=0.3, min_df=2,
                      sublinear_tf=True, dtype=np.float32)          # rows are L2-normalised: dot = cosine
Xtr = vec.fit_transform(tr.code.fillna("").astype(str).str.slice(0, 4000))
Xva = vec.transform(va.code.iloc[sel].fillna("").astype(str).str.slice(0, 4000))
XtrT = Xtr.T.tocsr()

sim = np.zeros(len(sel), dtype=np.float32); nn = np.zeros(len(sel), dtype=int); t0 = time.time()
for i in range(0, len(sel), 300):
    S = (Xva[i:i+300] @ XtrT).toarray()
    nn[i:i+300] = S.argmax(1); sim[i:i+300] = S.max(1)
    print(f"{min(i+300, len(sel))}/{len(sel)}  {time.time()-t0:.0f}s")

d = pd.DataFrame({"label": va.label.values[sel], "flag": (s2[sel] >= THR).astype(int), "sim": sim,
                  "nn_label": tr.label.values[nn], "project": va.project_key.values[sel]})
d["bucket"] = pd.cut(d.sim, [-0.01, 0.5, 0.8, 0.9, 0.95, 0.99, 1.01])
v, sf = d[d.label == 1], d[d.label == 0]

print("\nShare of val functions whose nearest TRAIN function has cosine >= 0.9")
print(f"  vulnerable: {(v.sim >= 0.9).mean():.1%}   safe (sample): {(sf.sim >= 0.9).mean():.1%}")
tab = pd.DataFrame({
    "n_vulnerable": v.groupby("bucket", observed=False).size(),
    "recall_CodeBERT": v.groupby("bucket", observed=False).flag.mean().round(3),
    "nn_train_is_vulnerable": v.groupby("bucket", observed=False).nn_label.mean().round(3),
    "n_safe_sample": sf.groupby("bucket", observed=False).size(),
    "false_alarm_rate": sf.groupby("bucket", observed=False).flag.mean().round(3)})
print("\nBy similarity of the nearest train function:")
print(tab)

d["chrome_linux"] = d.project.isin(["chrome", "linux"])
print("\nBy project group (chrome/linux vs other):")
print(d.groupby(["chrome_linux", "label"]).agg(n=("flag", "size"), flagged=("flag", "mean"), median_sim=("sim", "median")).round(3))
d.to_csv(f"{OUT}/neardup_check_val.csv", index=False)

300/4446  1s
600/4446  3s
900/4446  4s
1200/4446  4s
1500/4446  5s
1800/4446  5s
2100/4446  6s
2400/4446  7s
2700/4446  7s
3000/4446  8s
3300/4446  8s
3600/4446  9s
3900/4446  10s
4200/4446  10s
4446/4446  11s

Share of val functions whose nearest TRAIN function has cosine >= 0.9
  vulnerable: 18.5%   safe (sample): 12.2%

By similarity of the nearest train function:
              n_vulnerable  recall_CodeBERT  nn_train_is_vulnerable  n_safe_sample  false_alarm_rate
bucket                                                                                              
(-0.01, 0.5]           877            0.763                   0.113           1682             0.004
(0.5, 0.8]             250            0.736                   0.176            820             0.002
(0.8, 0.9]              52            0.750                   0.269            133             0.015
(0.9, 0.95]             55            0.636                   0.309            109             0.009
(0.95, 0.99]           1

In [ ]:
!pip -q install -r /content/AIES_PROJECT/requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/AIES_PROJECT'):
    !git clone https://github.com/AleezaMujahid/AIES_PROJECT /content/AIES_PROJECT
%cd /content/AIES_PROJECT
if not os.path.exists('data'):
    !ln -s DATA data
!pip -q install transformers accelerate

Mounted at /content/drive
Cloning into '/content/AIES_PROJECT'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 82 (delta 13), reused 77 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 51.97 MiB | 14.75 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/AIES_PROJECT


In [ ]:
#cellt1
import re, json, joblib, numpy as np, pandas as pd, torch
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix)
OUT = "/content/drive/MyDrive/codesentinel_ckpt_v2"
BEST = f"{OUT}/best"

def evaluate(y_true, p_vuln, threshold=0.5):
    y_true = np.asarray(y_true); p = np.asarray(p_vuln)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"threshold": float(threshold), "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "pr_auc": average_precision_score(y_true, p), "roc_auc": roc_auc_score(y_true, p),
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

def normalize_code(c):      # SAME function as in training
    return "\n".join(re.sub(r"[ \t]+", " ", l.strip()) for l in str(c).split("\n") if l.strip())

# ---- TEST SET: opened ONCE here ----
test = pd.read_csv("data/processed/bigvul_test.csv.gz", usecols=["sample_id", "project_key", "code", "label"])
test["label"] = test["label"].astype(int); y = test.label.values
test["code_n"] = test.code.map(normalize_code)
print("test rows", len(test), "| vulnerable", int(y.sum()), f"({y.mean():.1%})")

# ---- generic TF-IDF + LogReg (saved model, its own saved threshold) ----
TOK = r"[A-Za-z_][A-Za-z0-9_]*|\d+|->|==|!=|<=|>=|\+\+|--|&&|\|\||<<|>>|[^\sA-Za-z0-9_]"
bundle = joblib.load("models/baseline_generic/baseline.joblib")
bcfg = json.load(open("models/baseline_generic/config.json"))
tokre, keep = re.compile(TOK), bundle["keep"]
def generic_text(code):
    return " ".join(t if (not re.match(r"[A-Za-z_]", t) or t in keep) else "ID"
                    for t in tokre.findall(code[:bcfg["max_chars"]]))
texts = [generic_text(c) for c in test.code.fillna("").astype(str)]
p_tfidf = bundle["model"].predict_proba(bundle["vectorizer"].transform(texts))[:, 1]
print("TF-IDF generic scored. threshold", bcfg["threshold"])

test rows 27720 | vulnerable 1446 (5.2%)


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.9.1 when using version 1.6.1. This might lead to breaking c

AttributeError: 'LogisticRegression' object has no attribute 'multi_class'

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(BEST)

# was the function cut at 512 tokens? (needed later for the fit-vs-cut slice)
n_tok = np.array([len(x) for x in tok(list(test.code_n), add_special_tokens=False, truncation=False)["input_ids"]])
test["cut"] = (n_tok > 510).astype(int)
print(f"cut at 512 tokens: {test.cut.mean():.1%}")

@torch.no_grad()
def score(model, texts, bs=64):
    model.eval(); texts = np.asarray(texts, dtype=object)
    order = np.argsort([len(t) for t in texts]); out = np.zeros(len(texts))
    for i in range(0, len(texts), bs):
        idx = order[i:i+bs]
        enc = tok(list(texts[idx]), truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        out[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return out

m_after = AutoModelForSequenceClassification.from_pretrained(BEST).cuda()
p_after = score(m_after, test.code_n.values); del m_after
m_before = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2).cuda()
m_before.classifier.load_state_dict(torch.load(f"{OUT}/classifier_start.pt"))      # same random start as the "before" run
p_before = score(m_before, test.code_n.values); del m_before

THR_CB = json.load(open(f"{BEST}/codebert_config.json"))["threshold"]
rows = [("Majority guess", "before", evaluate(y, np.zeros(len(y)), 0.5)),
        ("TF-IDF + LogReg (generic)", "after (its saved threshold)", evaluate(y, p_tfidf, bcfg["threshold"])),
        ("CodeBERT", "before (start weights, thr 0.5)", evaluate(y, p_before, 0.5)),
        ("CodeBERT", "after (best epoch, frozen thr)", evaluate(y, p_after, THR_CB))]
res = pd.DataFrame([{"model": m, "stage": s, **r} for m, s, r in rows])
res.to_csv(f"{OUT}/test_results.csv", index=False)
test.assign(p_tfidf=p_tfidf, p_codebert_before=p_before, p_codebert_after=p_after)[
    ["sample_id", "project_key", "label", "cut", "p_tfidf", "p_codebert_before", "p_codebert_after"]
].to_csv(f"{OUT}/test_predictions.csv", index=False)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
print(res[["model", "stage", "threshold", "accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc"]].round(3))
print(res[["model", "stage", "TN", "FP", "FN", "TP"]])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


cut at 512 tokens: 13.0%


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


                       model                            stage  threshold  accuracy  precision  recall     f1  pr_auc  roc_auc
0             Majority guess                           before      0.500     0.948      0.000   0.000  0.000   0.052    0.500
1  TF-IDF + LogReg (generic)      after (its saved threshold)      0.637     0.882      0.163   0.305  0.212   0.159    0.720
2                   CodeBERT  before (start weights, thr 0.5)      0.500     0.948      0.000   0.000  0.000   0.045    0.466
3                   CodeBERT   after (best epoch, frozen thr)      0.660     0.881      0.169   0.327  0.223   0.151    0.740
                       model                            stage     TN    FP    FN   TP
0             Majority guess                           before  26274     0  1446    0
1  TF-IDF + LogReg (generic)      after (its saved threshold)  24006  2268  1005  441
2                   CodeBERT  before (start weights, thr 0.5)  26274     0  1446    0
3                   CodeBE

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
OUT = "/content/drive/MyDrive/codesentinel_ckpt_v2"
d = pd.read_csv(f"{OUT}/test_predictions.csv")
THR = {"tfidf": 0.6369718747240671,                                      # generic baseline's saved threshold
       "codebert": json.load(open(f"{OUT}/best/codebert_config.json"))["threshold"]}   # frozen on val
P = {"tfidf": "p_tfidf", "codebert": "p_codebert_after"}

def metrics(df):
    y = df.label.values; out = {"n": len(df), "%vuln": round(100*y.mean(), 1)}
    for m, col in P.items():
        p = df[col].values; pred = (p >= THR[m]).astype(int)
        out[f"{m} PR-AUC"] = round(average_precision_score(y, p), 3)
        out[f"{m} ROC"] = round(roc_auc_score(y, p), 3)
        out[f"{m} P"] = round(precision_score(y, pred, zero_division=0), 3)
        out[f"{m} R"] = round(recall_score(y, pred, zero_division=0), 3)
        out[f"{m} F1"] = round(f1_score(y, pred, zero_division=0), 3)
    return out

cl = d.project_key.isin(["chrome", "linux"])
slices = {"all test rows": d, "without chrome+linux": d[~cl], "only chrome+linux": d[cl],
          "fits in 512 tokens": d[d.cut == 0], "cut at 512 tokens": d[d.cut == 1]}
print(pd.DataFrame({k: metrics(v) for k, v in slices.items()}).T)

# Is the CodeBERT vs TF-IDF difference real? Bootstrap over test rows (500 resamples).
rng = np.random.RandomState(42); y = d.label.values; n = len(d)
dap, df1 = [], []
for _ in range(500):
    i = rng.randint(0, n, n); yy = y[i]
    if yy.sum() == 0: continue
    a = average_precision_score(yy, d.p_codebert_after.values[i]) - average_precision_score(yy, d.p_tfidf.values[i])
    b = f1_score(yy, (d.p_codebert_after.values[i] >= THR["codebert"]).astype(int), zero_division=0) - \
        f1_score(yy, (d.p_tfidf.values[i] >= THR["tfidf"]).astype(int), zero_division=0)
    dap.append(a); df1.append(b)
for name, v in [("PR-AUC", dap), ("F1", df1)]:
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"CodeBERT minus TF-IDF, {name}: mean {np.mean(v):+.3f}, 95% interval [{lo:+.3f}, {hi:+.3f}]"
          + ("   -> interval includes 0: no clear winner" if lo <= 0 <= hi else "   -> interval excludes 0"))

                            n  %vuln  tfidf PR-AUC  tfidf ROC  tfidf P  tfidf R  tfidf F1  codebert PR-AUC  codebert ROC  codebert P  codebert R  codebert F1
all test rows         27720.0    5.2         0.159      0.720    0.163    0.305     0.212            0.151         0.740       0.169       0.327        0.223
without chrome+linux   8410.0    6.2         0.238      0.772    0.189    0.508     0.275            0.227         0.786       0.201       0.498        0.287
only chrome+linux     19310.0    4.8         0.108      0.692    0.135    0.190     0.158            0.114         0.715       0.142       0.230        0.175
fits in 512 tokens    24103.0    3.8         0.077      0.667    0.106    0.118     0.111            0.082         0.697       0.110       0.154        0.128
cut at 512 tokens      3617.0   14.4         0.271      0.657    0.198    0.637     0.302            0.287         0.675       0.220       0.635        0.326
CodeBERT minus TF-IDF, PR-AUC: mean -0.007, 95% inte

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/AIES_PROJECT'):
    !git clone https://github.com/AleezaMujahid/AIES_PROJECT /content/AIES_PROJECT
%cd /content/AIES_PROJECT
if not os.path.exists('data'):
    !ln -s DATA data
!pip -q install -r requirements.txt
!pip -q install transformers accelerate

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/AIES_PROJECT
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, joblib, numpy as np, pandas as pd, torch
os.chdir('/content/AIES_PROJECT')
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
OUT = "/content/drive/MyDrive/codesentinel_ckpt_v2"; BEST = f"{OUT}/best"

def evaluate(y_true, p_vuln, threshold=0.5):
    y_true = np.asarray(y_true); p = np.asarray(p_vuln)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"threshold": float(threshold), "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "pr_auc": average_precision_score(y_true, p), "roc_auc": roc_auc_score(y_true, p),
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

def normalize_code(c):      # SAME function as in training
    return "\n".join(re.sub(r"[ \t]+", " ", l.strip()) for l in str(c).split("\n") if l.strip())

# ---- CVEfixes outside set (never used for training or tuning) ----
cols = ["sample_id", "project_key", "commit_id", "file", "func_name", "code", "label"]
cf = pd.read_csv("data/processed/cvefixes_external.csv.gz", usecols=cols)
cf["label"] = cf["label"].astype(int); cf[["file", "func_name"]] = cf[["file", "func_name"]].fillna("")
strict_ids = set(pd.read_csv("data/processed/cvefixes_external_strict.csv.gz", usecols=["sample_id"]).sample_id)
cf["strict"] = cf.sample_id.isin(strict_ids).astype(int)
cf["code_n"] = cf.code.map(normalize_code)
print("full", len(cf), f"({cf.label.mean():.1%} vulnerable) | strict", int(cf.strict.sum()),
      f"({cf.label[cf.strict==1].mean():.1%} vulnerable)")

# ---- generic TF-IDF + LogReg ----
TOK = r"[A-Za-z_][A-Za-z0-9_]*|\d+|->|==|!=|<=|>=|\+\+|--|&&|\|\||<<|>>|[^\sA-Za-z0-9_]"
bundle = joblib.load("models/baseline_generic/baseline.joblib")
bcfg = json.load(open("models/baseline_generic/config.json"))
tokre, keep = re.compile(TOK), bundle["keep"]
def generic_text(code):
    return " ".join(t if (not re.match(r"[A-Za-z_]", t) or t in keep) else "ID"
                    for t in tokre.findall(code[:bcfg["max_chars"]]))
texts = [generic_text(c) for c in cf.code.fillna("").astype(str)]
cf["p_tfidf"] = bundle["model"].predict_proba(bundle["vectorizer"].transform(texts))[:, 1]

# ---- CodeBERT (best epoch, frozen threshold) ----
tok = AutoTokenizer.from_pretrained(BEST)
@torch.no_grad()
def score(model, texts, bs=64):
    model.eval(); texts = np.asarray(texts, dtype=object)
    order = np.argsort([len(t) for t in texts]); out = np.zeros(len(texts))
    for i in range(0, len(texts), bs):
        idx = order[i:i+bs]
        enc = tok(list(texts[idx]), truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        out[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return out
m = AutoModelForSequenceClassification.from_pretrained(BEST).cuda()
cf["p_cb"] = score(m, cf.code_n.values); del m
cf.drop(columns=["code", "code_n"]).to_csv(f"{OUT}/cvefixes_predictions.csv", index=False)
print("scored and saved")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


full 11909 (39.9% vulnerable) | strict 5266 (49.9% vulnerable)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

scored and saved


In [ ]:
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
THR = {"tfidf": bcfg["threshold"], "codebert": json.load(open(f"{BEST}/codebert_config.json"))["threshold"]}
rows = []
for set_name, df in [("CVEfixes full", cf), ("CVEfixes strict", cf[cf.strict == 1])]:
    y = df.label.values
    rows.append({"set": set_name, "model": "Majority guess", **evaluate(y, np.zeros(len(y)), 0.5)})
    rows.append({"set": set_name, "model": "TF-IDF + LogReg (generic)", **evaluate(y, df.p_tfidf, THR["tfidf"])})
    rows.append({"set": set_name, "model": "CodeBERT (after)", **evaluate(y, df.p_cb, THR["codebert"])})
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/cvefixes_results.csv", index=False)
print("chance PR-AUC = vulnerable share: full %.3f, strict %.3f" % (cf.label.mean(), cf.label[cf.strict == 1].mean()))
print(res[["set", "model", "accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc"]].round(3))
print(res[["set", "model", "TN", "FP", "FN", "TP"]])

# ---- before-vs-after PAIR check: same function, same commit; label 1 = before the fix, 0 = after the fix ----
def pairs(df, col):
    out = []
    for _, x in df.groupby(["commit_id", "file", "func_name"]):
        if len(x) == 2 and set(x.label) == {0, 1}:
            out.append((x.loc[x.label == 1, col].iloc[0], x.loc[x.label == 0, col].iloc[0]))
    return np.array(out)
prow = []
for set_name, df in [("CVEfixes strict", cf[cf.strict == 1]), ("CVEfixes full", cf)]:
    for mname, col, t in [("TF-IDF + LogReg (generic)", "p_tfidf", THR["tfidf"]), ("CodeBERT (after)", "p_cb", THR["codebert"])]:
        a = pairs(df, col); b, f = a[:, 0], a[:, 1]
        prow.append({"set": set_name, "model": mname, "pairs": len(a),
                     "score before > after": round((b > f).mean(), 3),          # chance is about 0.5
                     "correct (before flagged, after not)": round(((b >= t) & (f < t)).mean(), 3),
                     "both flagged": round(((b >= t) & (f >= t)).mean(), 3),
                     "neither flagged": round(((b < t) & (f < t)).mean(), 3),
                     "reversed (after flagged, before not)": round(((b < t) & (f >= t)).mean(), 3)})
pr = pd.DataFrame(prow); pr.to_csv(f"{OUT}/cvefixes_pair_check.csv", index=False)
print("\nPAIR CHECK (one function: version before the fix vs after the fix)"); print(pr.T)

chance PR-AUC = vulnerable share: full 0.399, strict 0.499
               set                      model  accuracy  precision  recall     f1  pr_auc  roc_auc
0    CVEfixes full             Majority guess     0.601      0.000   0.000  0.000   0.399    0.500
1    CVEfixes full  TF-IDF + LogReg (generic)     0.523      0.398   0.379  0.389   0.401    0.506
2    CVEfixes full           CodeBERT (after)     0.526      0.396   0.358  0.376   0.392    0.501
3  CVEfixes strict             Majority guess     0.501      0.000   0.000  0.000   0.499    0.500
4  CVEfixes strict  TF-IDF + LogReg (generic)     0.491      0.488   0.422  0.453   0.493    0.490
5  CVEfixes strict           CodeBERT (after)     0.497      0.495   0.401  0.443   0.494    0.490
               set                      model    TN    FP    FN    TP
0    CVEfixes full             Majority guess  7155     0  4754     0
1    CVEfixes full  TF-IDF + LogReg (generic)  4426  2729  2950  1804
2    CVEfixes full           CodeBERT 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, shutil, numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
OUT = "/content/drive/MyDrive/codesentinel_ckpt_v2"
REP = "/content/AIES_PROJECT/reports"; os.makedirs(REP, exist_ok=True)

def md(df, cols=None, nd=3):
    df = df if cols is None else df[cols]
    f = lambda v: f"{v:.{nd}f}" if isinstance(v, float) else str(v)
    return "\n".join(["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)] +
                     ["| " + " | ".join(f(v) for v in r) + " |" for r in df.itertuples(index=False)])

M = ["accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc"]
val  = pd.read_csv(f"{OUT}/val_before_after.csv")
test = pd.read_csv(f"{OUT}/test_results.csv")
cve  = pd.read_csv(f"{OUT}/cvefixes_results.csv")
pair = pd.read_csv(f"{OUT}/cvefixes_pair_check.csv")
pred = pd.read_csv(f"{OUT}/test_predictions.csv")
cfg  = json.load(open(f"{OUT}/best/codebert_config.json"))
THR  = {"tfidf": 0.6369718747240671, "codebert": cfg["threshold"]}

# slices on the saved test predictions (analysis only, nothing is tuned)
def sl(df):
    y = df.label.values; o = {"rows": len(df), "% vulnerable": round(100*y.mean(), 1)}
    for m, col, nm in [("tfidf", "p_tfidf", "TF-IDF"), ("codebert", "p_codebert_after", "CodeBERT")]:
        p = df[col].values; q = (p >= THR[m]).astype(int)
        o[f"{nm} PR-AUC"] = round(average_precision_score(y, p), 3)
        o[f"{nm} recall"] = round(recall_score(y, q, zero_division=0), 3)
        o[f"{nm} F1"] = round(f1_score(y, q, zero_division=0), 3)
    return o
cl = pred.project_key.isin(["chrome", "linux"])
sdict = {"all test rows": pred, "without chrome+linux": pred[~cl], "only chrome+linux": pred[cl],
         "fits in 512 tokens": pred[pred.cut == 0], "cut at 512 tokens": pred[pred.cut == 1]}
slices = pd.DataFrame({k: sl(v) for k, v in sdict.items()}).T.reset_index().rename(columns={"index": "slice"})
slices["rows"] = slices["rows"].astype(int)

rng = np.random.RandomState(42); y = pred.label.values; n = len(pred); dap, df1 = [], []
for _ in range(500):
    i = rng.randint(0, n, n); yy = y[i]
    if yy.sum() == 0: continue
    dap.append(average_precision_score(yy, pred.p_codebert_after.values[i]) - average_precision_score(yy, pred.p_tfidf.values[i]))
    df1.append(f1_score(yy, (pred.p_codebert_after.values[i] >= THR["codebert"]).astype(int), zero_division=0) -
               f1_score(yy, (pred.p_tfidf.values[i] >= THR["tfidf"]).astype(int), zero_division=0))
ci = lambda v: f"mean {np.mean(v):+.3f}, 95% interval [{np.percentile(v, 2.5):+.3f}, {np.percentile(v, 97.5):+.3f}]"

text = f"""# Before vs after fine-tuning: results (Person B)

All numbers come from saved files in `codesentinel_ckpt_v2` (validation scores, test predictions, CVEfixes predictions).
Output is a **model score**, not a probability. Test set was opened once; the epoch and the threshold were chosen on validation only.

## 1. Setup
- CodeBERT (`microsoft/codebert-base`) + new 2-class head, max_length 512 (keep start), lr 2e-5, batch 16, {cfg['epochs']} epochs, seed {cfg['seed']}.
- Training subsample: {cfg['train_rows_used']:,} of {cfg['train_rows_full']:,} BigVul training rows (all {cfg['train_vulnerable_used']:,} vulnerable + {cfg['train_safe_used']:,} safe).
  Class weights {[round(w, 3) for w in cfg['class_weights']]} (balanced for the subsample).
- Input is normalised code: per-line strip, spaces/tabs collapsed, blank lines dropped (`normalize_code`). Frozen CodeBERT threshold {cfg['threshold']:.2f} (best F1 on validation). Best epoch: {cfg['best_epoch']}.
- Baseline: generic-token TF-IDF + Logistic Regression, trained on all 129,360 rows, its own saved threshold {THR['tfidf']:.3f}.

## 2. Validation (CodeBERT, before vs after training)
{md(val, ['stage', 'threshold_type'] + M)}

## 3. Test set (BigVul, 27,720 rows, opened once)
{md(test, ['model', 'stage', 'threshold'] + M)}

Confusion matrices:

{md(test, ['model', 'stage', 'TN', 'FP', 'FN', 'TP'])}

Accuracy of the majority guess is high only because about 94.8% of functions are safe; it finds no vulnerabilities.

## 4. Slices of the test set
{md(slices)}

CodeBERT minus TF-IDF, PR-AUC: {ci(dap)}
CodeBERT minus TF-IDF, F1: {ci(df1)}
(bootstrap over rows, 500 resamples; resampling commits instead would give wider intervals)

## 5. Outside test: CVEfixes (never used for training or tuning)
Vulnerable share is 39.9% (full) and 49.9% (strict), so chance PR-AUC is 0.40 and 0.50.

{md(cve, ['set', 'model'] + M)}

{md(cve, ['set', 'model', 'TN', 'FP', 'FN', 'TP'])}

## 6. Before-vs-after-fix pair check (same function, same commit)
Label 1 is the version before the fix, label 0 the version after. "score before > after" is about 0.5 for a model that cannot tell them apart.

{md(pair)}

## 7. Earlier run that exposed a shortcut (kept as a finding)
BigVul has a whitespace artifact: 56.1% of vulnerable but only 1.9% of safe training functions contain a line with trailing whitespace.
A first CodeBERT run on raw code reached validation PR-AUC 0.864 (F1 0.81). A model using only formatting features reached PR-AUC 0.632 on validation,
and after whitespace normalisation formatting plus length fell to 0.175. CodeBERT was therefore retrained on normalised code (section 2). The TF-IDF baseline ignores whitespace and was not affected.

## 8. Limitations to state in the report
- Training used a subsample ({cfg['train_rows_used']:,} of {cfg['train_rows_full']:,} rows); one run, one configuration, 2 epochs.
- Inputs longer than 512 tokens are cut (18.6% of raw training functions, 13.0% of normalised test functions); vulnerable functions are cut more often.
- Longer functions are more often vulnerable, so part of the score can come from length.
- BigVul labels are noisy and project-skewed (chrome and linux dominate). CVEfixes labels mean "before a fix", not proven vulnerable.
- Near-duplicate analysis was a spot check only (nearest-neighbour TF-IDF similarity; no clear effect on recall).
- Thresholds were chosen on validation; scores are not calibrated probabilities. "Not flagged" does not mean secure.
"""
open(f"{REP}/before_after.md", "w").write(text)
pd.concat([test.assign(dataset="BigVul test"), cve.rename(columns={"set": "dataset"})]).to_csv(f"{REP}/before_after.csv", index=False)
for f in ["val_before_after.csv", "test_results.csv", "cvefixes_results.csv", "cvefixes_pair_check.csv"]:
    shutil.copy(f"{OUT}/{f}", f"{REP}/{f}")
shutil.make_archive("/content/person_b_reports", "zip", REP)
print(text[:3500])
from google.colab import files
files.download("/content/person_b_reports.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
# Before vs after fine-tuning: results (Person B)

All numbers come from saved files in `codesentinel_ckpt_v2` (validation scores, test predictions, CVEfixes predictions).
Output is a **model score**, not a probability. Test set was opened once; the epoch and the threshold were chosen on validation only.

## 1. Setup
- CodeBERT (`microsoft/codebert-base`) + new 2-class head, max_length 512 (keep start), lr 2e-5, batch 16, 2 epochs, seed 42.
- Training subsample: 46,748 of 129,360 BigVul training rows (all 6,748 vulnerable + 40,000 safe).
  Class weights [0.584, 3.464] (balanced for the subsample).
- Input is normalised code: per-line strip, spaces/tabs collapsed, blank lines dropped (`normalize_code`). Frozen CodeBERT threshold 0.66 (best F1 on validation). Best epoch: 2.
- Baseline: generic-token TF-IDF + Logistic Regression, trained on all 129,360 rows, its

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
sys.argv = ["predict_codebert.py", "--model-dir", "/content/drive/MyDrive/codesentinel_ckpt_v2/best",
            "--code", "void f(char *s){ char b[8]; strcpy(b, s); }"]

In [ ]:
"""
predict_codebert.py -- score ONE C/C++ function with the fine-tuned CodeBERT (Person B -> Person C gate).

The model folder is the "best" folder from Drive (weights, tokenizer, codebert_config.json). It is NOT in GitHub.

Usage:
    python Scripts/predict_codebert.py --model-dir path/to/best --file my_function.c
    python Scripts/predict_codebert.py --model-dir path/to/best --code "void f(char *s){ char b[8]; strcpy(b, s); }"
    (with neither --file nor --code it reads the function from standard input)

The output is a MODEL SCORE, not a probability. "not flagged" does not mean the code is secure.
"""
import argparse
import json
import os
import re
import sys


def normalize_code(c):
    """Same preprocessing as in training: per-line strip, collapse spaces/tabs, drop blank lines."""
    return "\n".join(re.sub(r"[ \t]+", " ", l.strip()) for l in str(c).split("\n") if l.strip())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model-dir", required=True)
    ap.add_argument("--file")
    ap.add_argument("--code")
    a = ap.parse_args()
    cfg_path = os.path.join(a.model_dir, "codebert_config.json")
    if not os.path.exists(cfg_path):
        sys.exit(f"ERROR: {cfg_path} not found. --model-dir must be the 'best' folder.")
    if a.code is not None:
        code = a.code
    elif a.file:
        with open(a.file, encoding="utf-8", errors="replace") as f:
            code = f.read()
    else:
        code = sys.stdin.read()
    if not code.strip():
        sys.exit("ERROR: no code given.")

    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    cfg = json.load(open(cfg_path))
    max_len, thr = int(cfg["max_length"]), float(cfg["threshold"])
    tok = AutoTokenizer.from_pretrained(a.model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(a.model_dir)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()

    code_n = normalize_code(code)
    n_tok = len(tok(code_n, add_special_tokens=False, truncation=False)["input_ids"])
    enc = tok(code_n, truncation=True, max_length=max_len, return_tensors="pt").to(device)
    with torch.no_grad():
        score = float(torch.softmax(model(**enc).logits.float(), -1)[0, 1])
    verdict = "potentially vulnerable" if score >= thr else "not flagged"
    print(f"model score {score:.3f}  (threshold {thr:.2f})  ->  {verdict}")
    if n_tok > max_len - 2:
        print(f"WARNING: the input has {n_tok} tokens; only the first {max_len - 2} were analysed.")


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model score 0.325  (threshold 0.66)  ->  not flagged
